# Learning theory (gentle)

_Machine Learning_

**Why more data and simpler models generalize better.**

Learning theory asks: when will good training performance carry over to new data?

     Training error is how often you are wrong on data you already saw.

---

This notebook builds the idea up one step at a time: first measure the train-vs-validation gap as the dataset grows on synthetic data, then watch the same learning curve on real tumour scans. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

We measure how performance depends on dataset size in two steps: (1) build a synthetic dataset, then (2) compute a **learning curve** — train and validation accuracy at growing training-set sizes — and read off the generalisation gap.

### Step 1 — Build a synthetic dataset

`make_classification` gives 800 examples with 8 features, 5 of them genuinely informative. A fixed `random_state` keeps the run reproducible so the numbers below are stable.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve

# 800 examples, 8 features, 5 of them informative.
X, y = make_classification(n_samples=800, n_features=8, n_informative=5,
                           random_state=0)

### Step 2 — Compute the learning curve

`learning_curve` retrains the model on increasing fractions of the data and cross-validates each one. We print train accuracy, validation accuracy, and their **gap** — the gap shrinks as more data is added, which is exactly what learning theory predicts.

In [ ]:
# Train at growing dataset sizes, cross-validated 5 ways.
sizes, train_sc, val_sc = learning_curve(
    LogisticRegression(max_iter=500), X, y,
    train_sizes=[0.1, 0.3, 0.6, 1.0], cv=5, random_state=0)

# The gap between train and validation accuracy is the generalisation gap.
for n, tr, va in zip(sizes, train_sc.mean(1), val_sc.mean(1)):
    print("train size %4d  train acc %.3f  val acc %.3f  gap %.3f"
          % (n, tr, va, tr - va))

## Visualize the data & results

_On breast-cancer data, does adding more training examples close the gap between training and validation accuracy?_

### Step 1 — Load and standardise tumour scans

We load the real **breast-cancer** dataset and standardise the features so the logistic-regression solver converges cleanly. The goal is to see whether adding more training examples closes the train-vs-validation gap on real data too.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import learning_curve

bc = load_breast_cancer()

# Standardise features so the solver converges cleanly.
X = StandardScaler().fit_transform(bc.data)
y = bc.target

### Step 2 — Compute and plot the learning curve

We compute train and validation accuracy at growing training-set sizes, then plot both curves. As examples are added, the validation curve rises toward the training curve — the gap closing is generalisation improving with more data.

In [ ]:
sizes, train_sc, val_sc = learning_curve(
    LogisticRegression(max_iter=5000), X, y,
    train_sizes=[0.1, 0.3, 0.6, 1.0], cv=5)

plt.plot(sizes, train_sc.mean(1), color="#4ea1ff", marker="o", label="train accuracy")
plt.plot(sizes, val_sc.mean(1), color="#7ee787", marker="o", label="validation accuracy")
plt.xlabel("training examples")
plt.ylabel("accuracy")
plt.title("Learning curve")
plt.legend()
plt.show()